# Módulo 7 — TDS, clasificación y resultados

| | |
|---|---|
| **Responde** | Resultados/diagnóstico · metodología 7 y 9 · hipótesis 2 |
| **Código** | `clasificacion_resultados.py` (en esta misma carpeta) |
| **Recibe** | la tabla de características, una fila por imagen |
| **Devuelve** | métricas, figuras y tablas finales |
| **Responsable** | Lucas D'Elia |

> Las funciones se programan en `clasificacion_resultados.py`. Este cuaderno sólo las llama, grafica y discute.
> Para guardar una figura para el informe: `graficos.guardar_figura(fig, "nombre")`.
> Para comprobar el módulo: `python main.py --revisar` desde la carpeta del repo.

In [1]:
# Permite usar el código de las otras carpetas del repo
import sys
sys.path.insert(0, "../..")

# Recarga los cambios que hagas en los archivos .py sin reiniciar el cuaderno
%load_ext autoreload
%autoreload 2

import numpy as np
import pandas as pd
from scipy import stats

from modulos.herramientas import graficos, procesamiento_completo as pc
from modulos.m0_carga_de_datos import carga_de_datos
from modulos.m1_mejoramiento import mejoramiento
from modulos.m3_segmentacion import segmentacion
from modulos.m5_textura_criterio_d import textura_criterio_d
from modulos.m6_zonas_y_mapas import zonas_y_mapas
from modulos.m7_clasificacion_resultados import clasificacion_resultados as m7

graficos.aplicar_estilo()


## 1. Tabla de las 200 imágenes

Todavía no está M4, así que `main.py` no puede armar la tabla completa ni se puede calcular el TDS. Este cuaderno procesa las imágenes reales con M1, M3, M5 y M6 y evalúa todo lo que no necesita A, B y C. La tabla se guarda aparte, en `resultados/m7_tabla_sin_abc.csv`, para no confundirla con `caracteristicas.csv`, que es la completa.

También se segmenta cada imagen con los tres métodos de pelos, porque la hipótesis 1 los compara.


In [2]:
def procesar_sin_abc(caso):
    """Corre M1, M3, M5 y M6 sobre una imagen, sin pasar por M4.

    Entra: caso   Caso de M0
    Sale : tupla (Resultado, dict con la fracción de pelos, el V de la lesión y el Dice de cada método de pelos)
    """
    extra = {}
    for metodo in mejoramiento.METODOS_PELOS:
        mejorada, valida, pelos = mejoramiento.mejorar(caso.imagen, metodo)
        extra[f"dice_{metodo}"] = segmentacion.dice(segmentacion.segmentar(mejorada, valida), caso.mascara_ref)
    # el último de METODOS_PELOS es "cierre", el elegido: sigue con esa imagen
    res = pc.Resultado(caso=caso, img_entrada=caso.imagen, img_mejorada=mejorada, valida=valida, mascara_pelos=pelos)
    res.mascara = segmentacion.segmentar(mejorada, valida)
    res.dice = segmentacion.dice(res.mascara, caso.mascara_ref)
    v = mejoramiento.a_hsv(mejorada)[..., 2]
    res.textura = textura_criterio_d.atributos_glcm(v, res.mascara)
    res.zonas = zonas_y_mapas.contrastes_entre_zonas(zonas_y_mapas.atributos_por_zona(mejorada, res.mascara))
    extra["fraccion_pelos"] = pelos.mean()
    extra["V_lesion"] = np.median(v[res.mascara])
    return res, extra

resultados, extras = zip(*(procesar_sin_abc(carga_de_datos.cargar_caso(n)) for n in carga_de_datos.nombres("todos")))
tabla = pd.concat([pc.tabla_de_caracteristicas(resultados), pd.DataFrame(extras)], axis=1)
tabla = pc.completar_d(tabla)
tabla.to_csv("../../resultados/m7_tabla_sin_abc.csv", index=False)
entrenamiento, prueba = tabla[tabla.conjunto == "entrenamiento"], tabla[tabla.conjunto == "prueba"]
print(len(entrenamiento), "de entrenamiento y", len(prueba), "de prueba")
print("D se estima con", tabla.attrs["atributo_d"], "y los umbrales", np.round(tabla.attrs["umbrales_d"], 4))


140 de entrenamiento y 60 de prueba
D se estima con glcm_correlacion y los umbrales [0.8787 0.9516 0.9697 0.973  0.9794]


## 2. Segmentación

Resumen del método elegido en M3, con el Dice contra la máscara de PH2. Una imagen falla si su Dice es menor a 0,7.


In [3]:
resumen = pd.DataFrame([{"conjunto": n, "imágenes": len(t), "Dice medio": t.dice.mean(), "mediana": t.dice.median(),
                         "peor": t.dice.min(), "fallas": (t.dice < 0.7).sum()}
                        for n, t in (("entrenamiento", entrenamiento), ("prueba", prueba))])
graficos.guardar_tabla(resumen, "m7_dice", decimales=3)
resumen


,conjunto,imágenes,Dice medio,mediana,peor,fallas
0,entrenamiento,140,0.908343,0.931045,0.500659,2
1,prueba,60,0.916306,0.945954,0.299714,2


## 3. Hipótesis 1: quitar pelos con un cierre

La hipótesis dice que el cierre segmenta mejor las imágenes con pelos que no quitarlos y que un filtro de mediana. Una imagen tiene pelos si M1 marca más del 10 % de sus píxeles, el mismo criterio de M3. El Dice de cada imagen se compara de a pares con la prueba de Wilcoxon, porque son las mismas imágenes y las diferencias no tienen por qué ser normales.


In [4]:
filas = []
for nombre_conjunto, t in (("entrenamiento", entrenamiento), ("prueba", prueba)):
    pelos = t[t.fraccion_pelos > 0.10]
    for rival in ("ninguno", "mediana"):
        diferencia = pelos.dice_cierre - pelos[f"dice_{rival}"]
        filas.append({"conjunto": nombre_conjunto, "imágenes con pelos": len(pelos), "comparación": f"cierre - {rival}",
                      "Dice medio cierre": pelos.dice_cierre.mean(), "Dice medio rival": pelos[f"dice_{rival}"].mean(),
                      "mejora en": (diferencia > 0).sum(), "empeora en": (diferencia < 0).sum(),
                      "p de Wilcoxon": stats.wilcoxon(pelos.dice_cierre, pelos[f"dice_{rival}"]).pvalue})
hipotesis1 = pd.DataFrame(filas)
graficos.guardar_tabla(hipotesis1, "m7_hipotesis1", decimales=3)
hipotesis1


,conjunto,imágenes con pelos,comparación,Dice medio cierre,Dice medio rival,mejora en,empeora en,p de Wilcoxon
0,entrenamiento,30,cierre - ninguno,0.872836,0.892778,10,20,0.100397
1,entrenamiento,30,cierre - mediana,0.872836,0.897996,9,21,0.024786
2,prueba,9,cierre - ninguno,0.838430,0.840594,4,5,0.652344
3,prueba,9,cierre - mediana,0.838430,0.840868,3,6,0.359375


**Hipótesis 1: no se confirma.** En entrenamiento el cierre empeora más imágenes de las que mejora, tanto contra no quitar los pelos (20 contra 10, p = 0,10) como contra la mediana (21 contra 9, p = 0,025). En prueba, con sólo 9 imágenes con pelos, no hay diferencia detectable. El promedio cae sobre todo por cuatro imágenes de entrenamiento (IMD331, IMD418, IMD403 e IMD404), que pierden más de 0,1 de Dice; las medianas de los tres métodos casi coinciden (0,906, 0,913 y 0,906).

El cierre sigue siendo el método del algoritmo porque esa decisión se tomó antes de mirar prueba y porque, a diferencia de la mediana, no suaviza la red de pigmento que mide M5. Cambiarlo ahora invalidaría la evaluación en prueba.


## 4. Lo que se puede medir en prueba sin A, B y C

Los dos términos de zonas quedaron fijados en M6 y los umbrales de D salen de entrenamiento, así que esta es la primera vez que se miran estas 60 imágenes. El brillo de la lesión se agrega porque M6 mostró que el primer término lo sigue de cerca. El D del dermatólogo sirve de techo.

El puntaje parcial es el TDS extendido sin A, B ni C, o sea $0{,}5\\,D$ más los dos términos estandarizados con entrenamiento. **No es el TDS**, y sólo sirve para ver si las zonas agregan algo a D. Si A, B y C ya explican lo mismo, ese aporte puede desaparecer en el TDS real.


In [5]:
def estandarizar(t, columna):
    """Lleva una columna a la escala de entrenamiento.

    Entra: t        DataFrame con la columna
           columna  str
    Sale : array con (x - media) / desvío, con media y desvío de entrenamiento
    """
    return (t[columna].to_numpy(float) - entrenamiento[columna].mean()) / entrenamiento[columna].std()

def parcial(t):
    """Calcula 0,5 D más los dos términos de zonas, sin A, B ni C.

    Entra: t   DataFrame con D y las columnas de m7.TERMINOS_EXTRA
    Sale : array con un puntaje por fila
    """
    return m7.PESOS_TDS["D"] * t["D"].to_numpy(float) + sum(estandarizar(t, k) for k in m7.TERMINOS_EXTRA)

puntajes = {"D del dermatólogo": lambda t: t.D_derm.to_numpy(float),
            "D estimado": lambda t: t.D.to_numpy(float),
            "Periferia - centro, S": lambda t: t.zona_periferia_vs_centro_S.to_numpy(),
            "Piel - lesión, S": lambda t: t.zona_interfase_S.to_numpy(),
            "Oscuridad de la lesión (-V)": lambda t: -t.V_lesion.to_numpy(),
            "Puntaje parcial (0,5 D + zonas)": parcial}
real_e, real_p = entrenamiento.sospechosa.to_numpy(bool), prueba.sospechosa.to_numpy(bool)
filas = []
for nombre, puntaje in puntajes.items():
    valor, inferior, superior = m7.intervalo_auc(puntaje(prueba), real_p)
    filas.append({"medida": nombre, "AUC entrenamiento": m7.auc(puntaje(entrenamiento), real_e),
                  "AUC prueba": valor, "IC 95 % inf.": inferior, "IC 95 % sup.": superior})
auc_prueba = pd.DataFrame(filas)
graficos.guardar_tabla(auc_prueba, "m7_auc_prueba", decimales=2)
auc_prueba.round(3)


,medida,AUC entrenamiento,AUC prueba,IC 95 % inf.,IC 95 % sup.
0,D del dermatólogo,0.924,0.879,0.809,0.938
1,D estimado,0.730,0.604,0.462,0.742
2,"Periferia - centro, S",0.764,0.712,0.578,0.837
3,"Piel - lesión, S",0.685,0.602,0.459,0.747
4,Oscuridad de la lesión (-V),0.754,0.678,0.538,0.803
5,"Puntaje parcial (0,5 D + zonas)",0.831,0.714,0.581,0.844


In [6]:
# ¿Las zonas agregan algo a D? Misma muestra, diferencia de AUC con su intervalo
d_e, d_p = puntajes["D estimado"](entrenamiento), puntajes["D estimado"](prueba)
p_e, p_p = parcial(entrenamiento), parcial(prueba)
diferencia = m7.diferencia_auc(p_p, d_p, real_p)
print("AUC parcial - AUC D en prueba: %.3f (IC 95 %%: %.3f a %.3f)" % diferencia)

# Sensibilidad y especificidad en prueba, con el umbral elegido en entrenamiento
filas = []
for nombre, e, p in (("D estimado", d_e, d_p), ("Puntaje parcial", p_e, p_p)):
    umbral = m7.umbral_youden(e, real_e)
    filas.append({"medida": nombre, "umbral": umbral, **m7.metricas(p > umbral, real_p)})
clasificacion_parcial = pd.DataFrame(filas)
graficos.guardar_tabla(clasificacion_parcial, "m7_clasificacion_parcial", decimales=2)

# Acuerdo de D estimado con el dermatólogo en prueba
print("D en prueba: acierto exacto %.3f, error medio %.3f, kappa %.3f" % (
    (prueba.D == prueba.D_derm).mean(), (prueba.D - prueba.D_derm).abs().mean(), m7.kappa_cohen(prueba.D, prueba.D_derm)))
print("Decir siempre la mediana de entrenamiento: acierto %.3f" % (prueba.D_derm == entrenamiento.D_derm.median()).mean())
print("rho de los términos con el Dice y con V en prueba:")
print(prueba[["zona_periferia_vs_centro_S", "zona_interfase_S", "dice", "V_lesion"]].corr("spearman").round(2).to_string())
clasificacion_parcial.round(3)


AUC parcial - AUC D en prueba: 0.110 (IC 95 %: -0.008 a 0.225)
D en prueba: acierto exacto 0.400, error medio 0.983, kappa 0.168
Decir siempre la mediana de entrenamiento: acierto 0.500
rho de los términos con el Dice y con V en prueba:
                            zona_periferia_vs_centro_S  zona_interfase_S  dice  V_lesion
zona_periferia_vs_centro_S                        1.00              0.04  0.14     -0.67
zona_interfase_S                                  0.04              1.00 -0.42     -0.21
dice                                              0.14             -0.42  1.00     -0.06
V_lesion                                         -0.67             -0.21 -0.06      1.00


,medida,umbral,sensibilidad,especificidad,vpp,exactitud
0,D estimado,1.500,0.556,0.625,0.690,0.583
1,Puntaje parcial,0.549,0.806,0.542,0.725,0.700


**Qué pasó en prueba.** D estimado cae de AUC 0,73 en entrenamiento a 0,60, y su intervalo (0,46 a 0,74) incluye al 0,5, así que en prueba no se puede afirmar que separe los grupos. El D del dermatólogo también baja (de 0,92 a 0,88), pero sigue muy arriba. Además, D estimado acierta el valor exacto en el 40 % de las imágenes, contra el 50 % de decir siempre la mediana, con κ = 0,17.

La saturación periferia − centro es lo único medido automáticamente cuyo intervalo no incluye al 0,5 (AUC 0,71, de 0,58 a 0,84). Con 60 imágenes el intervalo es ancho, y no se distingue del brillo de la lesión solo (0,68), al que sigue de cerca (ρ = −0,67). La saturación piel − lesión baja a 0,60 y su intervalo incluye al 0,5.

Sumar las zonas a D sube el AUC en 0,11, pero el intervalo de esa diferencia (−0,01 a 0,23) incluye al 0, de modo que la mejora no es significativa. Con el umbral de entrenamiento, el puntaje parcial tiene sensibilidad 0,81 y especificidad 0,54 en prueba. Nada de esto responde la hipótesis 2, que depende de A, B y C.


## 5. TDS e hipótesis 2

Esta parte necesita A, B y C de M4. Cuando esté, `python main.py` arma `resultados/caracteristicas.csv` y `m7.generar(tabla)` guarda la tabla final, el acuerdo de A y C, la curva ROC y la distribución del TDS por grupo.
